In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
# ---------------------------------------------------------------------------
# Read source tables with row-count validation
# ---------------------------------------------------------------------------

dim_products_df  = spark.read.table("az_ci_de_dbs.ecom_dev.dim_products")
dim_customers_df = spark.read.table("az_ci_de_dbs.ecom_dev.dim_customers")
fact_orders_df   = spark.read.table("az_ci_de_dbs.ecom_dev.fact_orders")

validate_row_count(dim_products_df,  min_rows=1, label="dim_products")
validate_row_count(dim_customers_df, min_rows=1, label="dim_customers")
validate_row_count(fact_orders_df,   min_rows=1, label="fact_orders")

logger.info(
    "Source tables loaded – products=%d, customers=%d, orders=%d",
    dim_products_df.count(), dim_customers_df.count(), fact_orders_df.count(),
)

In [0]:
# ---------------------------------------------------------------------------
# Enriched Orders
# Performance: broadcast() the smaller dimension tables to avoid
#              shuffle joins on the larger fact table.
# Assumption: dim tables are small enough to fit in driver memory.
# ---------------------------------------------------------------------------

enriched_orders = (
    fact_orders_df.alias("o")
    .join(F.broadcast(dim_customers_df).alias("c"), on="customer_id", how="inner")
    .join(F.broadcast(dim_products_df).alias("p"),  on="product_id",  how="inner")
    .select(
        "o.*",
        "c.customer_name",
        "c.country",
        "p.category",
        "p.sub_category",
    )
)

enriched_orders = enriched_orders.dropDuplicates()

# Data quality gate
validate_row_count(enriched_orders, min_rows=1, label="enriched_orders")

delta_writer(enriched_orders, "az_ci_de_dbs.ecom_dev.enriched_orders")
logger.info("enriched_orders written – %d rows", enriched_orders.count())

In [0]:
# ---------------------------------------------------------------------------
# Profit By Year / Category / Customer
# Reads from the just-written enriched_orders for consistency.
# ---------------------------------------------------------------------------

profit_by_year = (
    spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
    .withColumn("year", F.year(F.col("order_date")))
    .groupBy("year", "category", "sub_category", "customer_name")
    .agg(F.sum("profit").alias("total_profit"))
)

validate_row_count(profit_by_year, min_rows=1, label="profit_by_year")

delta_writer(profit_by_year, "az_ci_de_dbs.ecom_dev.profit_by_year")
logger.info("profit_by_year written %d rows", profit_by_year.count())

In [0]:
%sql
-- Profit by Year
select year, sum(total_profit) as profit 
from az_ci_de_dbs.ecom_dev.profit_by_year 
group by year 
order by year;

-- Profit by Year + Product Category
select year, category, sum(total_profit) as profit 
from az_ci_de_dbs.ecom_dev.profit_by_year 
group by year, category 
order by year, category;

-- Profit by Customer
select customer_name as customer, sum(total_profit) as profit 
from az_ci_de_dbs.ecom_dev.profit_by_year 
group by customer_name 
order by customer_name;

-- Profit by Customer + Year
select year, customer_name as customer, sum(total_profit) as profit 
from az_ci_de_dbs.ecom_dev.profit_by_year 
group by year, customer_name 
order by year, customer_name;

